# 05 - Bootstrap confidence intervals and paired differences

Mirrors `notebooks_withseed/04_bootstrap_ci`: 95% bootstrap intervals on the pooled out-of-fold
predictions, plus paired differences between methods computed on identical resamples.

**One limitation specific to this cohort.** The handcrafted baseline exists only for the 148 OSF
subjects; Pooja's cohort has no equivalent scalar metrics. So handcrafted comparisons are restricted
to the OSF subset, while frozen and fine-tuned comparisons use whichever cohort is loaded.

In [ ]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')
import numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
OSF, OUT = '../data/preprocessed', '../results_parcel'
MODEL, SUFFIX = 'tiny', 'cortical'
cv = StratifiedKFold(5, shuffle=True, random_state=123)

# Use the COMBINED frozen features so every method here covers the same subjects.
# Mixing the 161-subject parcel cohort with 309-subject fine-tune results in one chart
# would compare numbers computed on different people.
Xc_path = f'{OUT}/combined_frozen_{MODEL}_{SUFFIX}.npy'
assert os.path.exists(Xc_path), 'run notebook 03 first (it writes the combined frozen features)'
X_comb = np.load(Xc_path)
osf_y  = np.load(f'{OSF}/labels.npy')
par_y  = np.load(f'{OUT}/parcel_labels_{SUFFIX}.npy')
labels = np.concatenate([osf_y, par_y])
N = len(labels)
assert len(X_comb) == N, f'feature/label mismatch: {len(X_comb)} vs {N}'
print(f'combined cohort: {X_comb.shape} | {N} subjects '
      f'(OSF {len(osf_y)} + parcel {len(par_y)})')

def oof_lr(X, y):
    pred, prob = np.zeros(len(y), int), np.zeros(len(y))
    for tr, te in cv.split(X, y):
        m = LogisticRegression(max_iter=2000).fit(X[tr], y[tr])
        pred[te], prob[te] = m.predict(X[te]), m.predict_proba(X[te])[:, 1]
    return pred, prob

M = []
p, pr = oof_lr(X_comb, labels); M.append(('Frozen combined + LR', p, pr, labels))

ft = f'{OUT}/finetune_combined_{MODEL}.pt'
if os.path.exists(ft):
    d = torch.load(ft, map_location='cpu', weights_only=False)
    y_all = d['labels']
    for nf, r in sorted(d.get('cv', {}).items(), reverse=True):
        M.append((f'Fine-tune combined last-{11-nf} + NN', r['nn_pred'], r['nn_prob'], y_all))
        M.append((f'Fine-tune combined last-{11-nf} + LR', r['lr_pred'], r['lr_prob'], y_all))
else:
    print('note: no fine-tune checkpoint yet, run notebook 04 first')
print(f'{len(M)} methods assembled')

In [ ]:
def boot(pred, y, metric, nb=2000, seed=123):
    rng = np.random.default_rng(seed); idx = np.arange(len(y)); v = []
    for _ in range(nb):
        b = rng.choice(idx, len(y), replace=True)
        try: v.append(metric(y[b], pred[b]))
        except ValueError: pass
    return metric(y, pred), *np.percentile(v, [2.5, 97.5])

rows = []
for name, pred, prob, y in M:
    a, alo, ahi = boot(pred, y, accuracy_score)
    u, ulo, uhi = boot(prob, y, roc_auc_score)
    rows.append((name, a, alo, ahi, u, ulo, uhi))
    print(f'{name:38s} acc {a:.3f} [{alo:.3f},{ahi:.3f}]  AUC {u:.3f} [{ulo:.3f},{uhi:.3f}]')

In [ ]:
# Paired differences against the first method, on identical resamples.
def boot_diff(a, b, y, metric, nb=2000, seed=123):
    rng = np.random.default_rng(seed); idx = np.arange(len(y)); d = []
    for _ in range(nb):
        s = rng.choice(idx, len(y), replace=True)
        try: d.append(metric(y[s], a[s]) - metric(y[s], b[s]))
        except ValueError: pass
    return metric(y, a) - metric(y, b), *np.percentile(d, [2.5, 97.5])

if len(M) > 1:
    ref_name, ref_pred, ref_prob, ref_y = M[0]
    print(f'Paired differences vs {ref_name}\n')
    for name, pred, prob, y in M[1:]:
        if len(y) != len(ref_y):
            print(f'{name:38s} skipped (different cohort size)'); continue
        da, lo, hi = boot_diff(pred, ref_pred, y, accuracy_score)
        sig = 'YES' if (lo > 0 or hi < 0) else 'no'
        print(f'{name:38s} D acc {da:+.3f} [{lo:+.3f},{hi:+.3f}]  sig? {sig}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 1 + 0.5*len(rows)))
y = np.arange(len(rows))
for ax, (vi, lo_i, hi_i), lab in ((ax1, (1,2,3), 'Accuracy'), (ax2, (4,5,6), 'AUC')):
    v  = [r[vi] for r in rows]
    el = [r[vi]-r[lo_i] for r in rows]; eh = [r[hi_i]-r[vi] for r in rows]
    ax.barh(y, v, xerr=[el, eh], capsize=4, color='#4E79A7', error_kw=dict(lw=1, ecolor='#444'))
    ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows] if ax is ax1 else [])
    ax.set_xlim(0.4, 1.0); ax.set_xlabel(lab); ax.grid(axis='x', alpha=0.3); ax.set_axisbelow(True)
    ax.set_title(f'{lab} (95% bootstrap CI)')
plt.tight_layout(); plt.savefig(f'{OUT}/parcel_bootstrap_comparison.png', dpi=150, bbox_inches='tight'); plt.show()
print('saved parcel_bootstrap_comparison.png')